# 06_symbolic_forecasting.ipynb

This notebook builds a symbolic polynomial forecasting model using SageMath. You can experiment with:
- Degree of the polynomial
- Number of training samples
- Prediction horizon
- Evaluation accuracy


## 1. Setup and Data Load

In [1]:
from sage.all import *
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

# Load BTC time series
df = pd.read_csv('../data/bitcoin_timeseries.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').set_index('timestamp')


## 2. Define Hyperparameters

In [2]:
# Hyperparameters
days_to_use = 7          # trailing days of data
resample_freq = '1h'     # hourly
samples_to_use = 100     # how many points from tail to use
degree = 5               # degree of the polynomial
forecast_hours = 12      # prediction horizon


## 3. Prepare Training Data

In [3]:
# 1. Select recent data
end_time = df.index.max()
start_time = end_time - pd.Timedelta(days=int(days_to_use))  # Convert Sage int
recent = df.loc[(df.index >= start_time) & (df.index <= end_time)]

# 2. Resample and clean
df_model = recent.resample(resample_freq).mean().dropna().reset_index()

# 3. Select last N samples
df_model = df_model.tail(int(samples_to_use))  # Convert Sage int

# 4. Normalize time axis (in hours from first point, centered)
timestamps = df_model['timestamp']
t0 = timestamps.iloc[int(0)]  # force cast, just in case
x_vals = [(t - t0).total_seconds() / int(3600) for t in timestamps]  # Use 3600 as Python int
x_mean = sum(x_vals) / len(x_vals)
x_vals = [x - x_mean for x in x_vals]

# 5. Target values
y_vals = df_model['price_usd'].tolist()

## 4. Symbolic Polynomial Fitting

In [4]:
# Define symbolic model
x = var('x')
params = list(var([f'a{i}' for i in range(degree + 1)]))
model = sum(params[i] * x**i for i in range(degree + 1))

# Fit the model
points = list(zip(x_vals, y_vals))
fit = find_fit(points, model, parameters=params, variables=[x], solution_dict=True)
model_fitted = model.subs(fit)

# Evaluate on training data
f = lambda t: float(model_fitted.subs(x=t))
y_fit = [f(t) for t in x_vals]


## 5. Forecast Future Values

In [6]:
# Ensure forecast horizon is Python int
forecast_horizon = int(forecast_hours)

# 1. Generate future x values
last_x = float(x_vals[-1])
future_x = [last_x + i for i in range(1, forecast_horizon + 1)]
future_preds = [f(float(t)) for t in future_x]

# 2. Generate future timestamps
last_time = df_model['timestamp'].iloc[-1]  # Using Python int -1 instead of -Integer(1)
future_times = [last_time + timedelta(hours=int(i)) for i in range(1, forecast_horizon + 1)]

# 3. Build forecast DataFrame
df_future = pd.DataFrame({
    'timestamp': future_times,
    'predicted_price': future_preds
})

TypeError: Cannot index by location index with a non-integer key

## 6. Plot Historical and Forecasted Values

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df_model['timestamp'], y_vals, label='Historical Price', marker='o')
plt.plot(df_model['timestamp'], y_fit, label='Fitted Curve', linestyle='--')
plt.plot(df_future['timestamp'], df_future['predicted_price'], label='Forecast', marker='x', linestyle='--')
plt.title(f"BTC Price Forecast using Degree-{degree} Symbolic Polynomial")
plt.xlabel("Time")
plt.ylabel("Price (USD)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('../reports/symbolic_forecast_polydeg{}.png'.format(degree))
plt.show()


## 7. Evaluate Fit Accuracy on Training Data

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_vals, y_fit)
rmse = mean_squared_error(y_vals, y_fit, squared=False)

print(f"📏 MAE (Mean Absolute Error): {mae:.2f} USD")
print(f"📏 RMSE (Root Mean Squared Error): {rmse:.2f} USD")
